# 02 - Modelado inicial

Líneas base reproducibles para regresión y clasificación. No se incluyen métricas precalculadas: las celdas deben ejecutarse con los CSV locales. Se usa una partición cronológica 80/20 y todo el preprocesamiento se ajusta solo en entrenamiento.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    balanced_accuracy_score, classification_report, confusion_matrix,
    f1_score, mean_absolute_error, precision_score, r2_score,
    recall_score, roc_auc_score, root_mean_squared_error,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import build_order_level_dataset, load_olist_tables, modeling_frame

tables = load_olist_tables(ROOT / 'data' / 'raw')
orders = build_order_level_dataset(tables)

## Funciones compartidas

La partición conserva el orden temporal. La mediana, moda, escalado y categorías del one-hot se aprenden únicamente con el 80% inicial.

In [ ]:
def chronological_split(X, y, dates, train_fraction=0.8):
    cut = int(len(X) * train_fraction)
    if cut <= 0 or cut >= len(X):
        raise ValueError('No hay suficientes filas para una partición temporal.')
    return (
        X.iloc[:cut], X.iloc[cut:],
        y.iloc[:cut], y.iloc[cut:],
        dates.iloc[:cut], dates.iloc[cut:],
    )


def make_preprocessor(X):
    categorical = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    numeric = X.columns.difference(categorical).tolist()
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=10)),
    ])
    return ColumnTransformer([
        ('numeric', numeric_pipe, numeric),
        ('categorical', categorical_pipe, categorical),
    ])

## 1. Línea base de regresión: Ridge

Se reportan MAE, RMSE y $R^2$ solo cuando esta celda se ejecuta. No interprete resultados antes de revisar la cobertura temporal y los valores extremos.

In [ ]:
X_reg, y_reg, dates_reg = modeling_frame(orders, 'delivery_time_days')
Xr_train, Xr_test, yr_train, yr_test, dr_train, dr_test = chronological_split(
    X_reg, y_reg, dates_reg
)
print(f'Entrenamiento: {dr_train.min()} a {dr_train.max()} ({len(Xr_train):,} pedidos)')
print(f'Prueba: {dr_test.min()} a {dr_test.max()} ({len(Xr_test):,} pedidos)')

regression_model = Pipeline([
    ('preprocess', make_preprocessor(Xr_train)),
    ('model', Ridge(alpha=1.0, solver='lsqr')),
])
regression_model.fit(Xr_train, yr_train)
reg_pred = regression_model.predict(Xr_test)

regression_metrics = pd.Series({
    'MAE_days': mean_absolute_error(yr_test, reg_pred),
    'RMSE_days': root_mean_squared_error(yr_test, reg_pred),
    'R2': r2_score(yr_test, reg_pred),
}, name='Ridge temporal holdout')
regression_metrics

## 2. Línea base de clasificación: regresión logística

Se ponderan las clases para una primera línea base. La exactitud simple no se usa como métrica principal ante posible desbalance.

In [ ]:
X_cls, y_cls, dates_cls = modeling_frame(orders, 'late_delivery')
Xc_train, Xc_test, yc_train, yc_test, dc_train, dc_test = chronological_split(
    X_cls, y_cls, dates_cls
)
if yc_train.nunique() < 2 or yc_test.nunique() < 2:
    raise ValueError('La partición temporal no contiene ambas clases; revise el corte.')
print(f'Entrenamiento: {dc_train.min()} a {dc_train.max()} ({len(Xc_train):,} pedidos)')
print(f'Prueba: {dc_test.min()} a {dc_test.max()} ({len(Xc_test):,} pedidos)')
print('Prevalencia de retraso:', y_cls.groupby(dates_cls.ge(dc_test.min())).mean().rename({False: 'train', True: 'test'}))

classification_model = Pipeline([
    ('preprocess', make_preprocessor(Xc_train)),
    ('model', LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    )),
])
classification_model.fit(Xc_train, yc_train)
cls_pred = classification_model.predict(Xc_test)
cls_prob = classification_model.predict_proba(Xc_test)[:, 1]

classification_metrics = pd.Series({
    'balanced_accuracy': balanced_accuracy_score(yc_test, cls_pred),
    'precision': precision_score(yc_test, cls_pred, zero_division=0),
    'recall': recall_score(yc_test, cls_pred, zero_division=0),
    'F1': f1_score(yc_test, cls_pred, zero_division=0),
    'ROC_AUC': roc_auc_score(yc_test, cls_prob),
}, name='LogisticRegression temporal holdout')
classification_metrics

In [ ]:
print(classification_report(yc_test, cls_pred, digits=3, zero_division=0))
pd.DataFrame(
    confusion_matrix(yc_test, cls_pred),
    index=['real_a_tiempo', 'real_tarde'],
    columns=['pred_a_tiempo', 'pred_tarde'],
)

## Siguientes pasos para el Entregable II

- Definir validación temporal interna y una malla de hiperparámetros.
- Comparar las cinco familias exigidas: paramétrica, no paramétrica, ensamble de árboles, red neuronal y SVM.
- Estimar intervalos de confianza y separar resultados de entrenamiento, validación y prueba.
- Analizar variables y evaluar PCA y UMAP sobre los dos mejores modelos.
- Fijar semillas y registrar tiempos/costo computacional.